In [ ]:
! pip install pandas numpy scikit-learn  shap matplotlib
! pip install shap  
! pip install lightgbm

'pip' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


In [2]:
import pandas as pd
from constantes import (
    ENDERECO_DATASET_PENSE,
    ENDERECO_DICIONARIO_MICRODADOS_PENSE,
    ENDERECO_GERACAO_RESUMO_SIGNIFICADO_VARIAVEIS,
    ENDERECO_GERACAO_RESUMO_SIGNIFICADO_VARIAVEIS_DEPENDENTES,
    ENDERECO_GERACAO_RESUMO_SIGNIFICADO_VARIAVEIS_INDEPENDENTES,
    VARIAVEIS_DEPENDENTES,
    VARIAVEIS_INDEPENDENTES,
    NUMERO_CLASSES_POR_VARIAVEL_DEPENDENTE,
    NOVA_LINHA,
    COLUNA_PESO
)

In [3]:
#  Carregar o dataset normalizado
caminho = "../dados/em_processamento/dados_preparados_v2.txt"
dados_normalizados = pd.read_csv(caminho, sep=";", low_memory=False)

In [5]:
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import train_test_split


# Definir variável-alvo (Ex: 1 = sente-se sozinho frequentemente, 0 = raramente/nunca)
# Substitua 'VAR_ALVO' pelo código real da pergunta no dicionário do IBGE
dados_normalizados = dados_normalizados.dropna(subset=['B12009'])
y = dados_normalizados['B12009'].apply(lambda x: 1 if x in [4, 5] else 0)

# Definir variáveis explicativas (Ex: bullying, horas de sono, atividade física) e o peso amostral
colunas_explicativas = VARIAVEIS_INDEPENDENTES
X = dados_normalizados[colunas_explicativas].copy()
pesos = dados_normalizados['PESO_ALUNO']  # Coluna de peso amostral da PeNSE

# Converter variáveis categóricas para o tipo 'category' (o LightGBM lê nativamente)
for col in colunas_explicativas:
  X[col] = X[col].astype('category')

# 2. Dividir em treino e teste (mantendo os pesos alinhados)
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, pesos, test_size=0.2, random_state=42
)

# 3. Treinar o modelo LightGBM considerando o peso amostral do IBGE
modelo = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42)
modelo.fit(X_train, y_train, sample_weight=w_train)

# 4. Extrair os SHAP Values
explainer = shap.TreeExplainer(modelo)

# Como a base da PeNSE é grande, pegue uma amostra (ex: 5.000 linhas) para rodar rápido
X_amostra = X_test.sample(n=5000, random_state=42)
shap_values = explainer(X_amostra)

# 5. Visualizar os resultados (Gráfico Beeswarm)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_amostra)

ModuleNotFoundError: No module named 'shap'